In [1]:
import pandas as pd
import yfinance as yf
import numpy as np

### Data

In [2]:
df = pd.read_csv("nq-15min.csv")

df.set_index("Datetime", inplace=True)

df = df[["Open", "High", "Low", "Close"]]

df.index = (
    pd.to_datetime(df.index, unit='ms', utc=True).tz_convert('America/New_York')
)

In [3]:
# Ticker = "NQ=F"

# df = yf.download(Ticker, period="max", interval="15m")

# df.columns.names = [None, None]

# df.columns = df.columns.get_level_values(0)

# df = df.drop(columns=["Volume"])

In [4]:
df

,Open,High,Low,Close
Datetime,,,,
2019-12-31 19:00:00-05:00,8750.160,8750.160,8750.160,8750.160
2019-12-31 19:15:00-05:00,8750.160,8750.160,8750.160,8750.160
2019-12-31 19:30:00-05:00,8750.160,8750.160,8750.160,8750.160
2019-12-31 19:45:00-05:00,8750.160,8750.160,8750.160,8750.160
2019-12-31 20:00:00-05:00,8750.160,8750.160,8750.160,8750.160
...,...,...,...,...
2026-08-04 17:45:00-04:00,29787.043,29787.043,29787.043,29787.043
2026-08-04 18:00:00-04:00,29674.810,29711.131,29652.487,29709.965
2026-08-04 18:15:00-04:00,29709.632,29718.589,29685.543,29707.531


### Plot Data

In [5]:
# plot

### Vars

In [6]:
RANGE_LENGTH = 1
RRR = 2
TARGET_CANDLE_TIME = "09:15"
SL_RANGE_PCT = 2

SL_DISTANCE_R = 0.5
TRAIL_START_R = 1

MAX_LOOKAHEAD = 25

### Strategy

In [ ]:
def compute_range(high, low, root, range_length, eps = 0):
    range_high = high[root - range_length + 1:root + 1].max() + eps
    range_low = low[root - range_length + 1:root + 1].min() - eps
    return range_high, range_low

In [ ]:
# def simulate_trade(
#     high,
#     low,
#     close,
#     start_i,
#     range_high,
#     range_low,
#     rrr,
#     trail_start_r=None,
#     trail_distance_r=None,
#     max_lookahead=None,
# ):
#     delta = range_high - range_low

#     _trades = []

#     is_long = False
#     is_short = False

#     LONG_TRADE = 1
#     SHORT_TRADE = -1

#     upper_target = range_high + delta
#     lower_target = range_low - delta

#     end_i = len(high) if max_lookahead is None else min(len(high), start_i + max_lookahead)

#     for i in range(start_i, end_i):
#         if len(_trades) == 0:
#             if high[i] >= range_high and low[i] > range_low: _trades.append(LONG_TRADE)
#             elif low[i] <= range_low and high[i] < range_high: _trades.append(SHORT_TRADE)
#             elif low[i] > range_low and high[i] < range_high: continue
#             else: return -1

#         else:
            
#         if is_long:
#             if low[i] <= sl: return (sl - entry) / delta

#             if high[i] - range_high > rrr * delta: return rrr

#             if trail_start_r is not None:
#                 best_price = max(best_price, high[i])
#                 profit_r = (best_price - entry) / delta
#                 if profit_r >= trail_start_r:
#                     sl = max(sl, best_price - trail_distance_r * delta)

#         elif is_short:
#             if high[i] >= sl: return (entry - sl) / delta

#             if range_low - low[i] > rrr * delta: return rrr

#             if trail_start_r is not None:
#                 best_price = min(best_price, low[i])
#                 profit_r = (entry - best_price) / delta
#                 if profit_r >= trail_start_r:
#                     sl = min(sl, best_price + trail_distance_r * delta)

#     # loop exhausted (either max_lookahead or end of data)
#     if is_long or is_short:
#         exit_price = close[end_i - 1]           # mark-to-market at the cutoff bar
#         outcome_r = (exit_price - entry) / delta if is_long else (entry - exit_price) / delta
#         return outcome_r

#     return None  # no trade was triggered within the lookahead window

In [9]:
def run_range_breakout(
    df,
    sl_range_pct=SL_RANGE_PCT,
    range_length=RANGE_LENGTH,
    rrr=RRR,
    target_candle_time=TARGET_CANDLE_TIME,
    balance=1000,
    risk_per_trade=10,
    trail_start_r=None,
    trail_distance_r=None,
    max_lookahead=None, #--
):
    high = df["High"].to_numpy()
    low = df["Low"].to_numpy()
    close = df["Close"].to_numpy() #--

    PnL = np.full(len(df), 0.0)
    cum_pnl = 0.0

    for root in range(range_length - 1, len(df)):
        if balance + cum_pnl * risk_per_trade < 0: break

        if df.index[root].strftime("%H:%M") != target_candle_time: continue

        range_high, range_low = compute_range(high, low, root, range_length)

        outcome = simulate_trade(
            high,
            low,
            close,
            root + 1,
            range_high,
            range_low,
            sl_range_pct,
            rrr,
            trail_start_r=trail_start_r,
            trail_distance_r=trail_distance_r,
            max_lookahead=max_lookahead,
        )

        if outcome is not None:
            PnL[root] = outcome
            cum_pnl += outcome

    return pd.DataFrame(
        {
            "PnL": PnL,
            "Cumulative_PnL": np.cumsum(PnL),
            "Balance": balance + np.cumsum(PnL) * risk_per_trade,
        },
        index=df.index,
    )

In [10]:
result = run_range_breakout(df, trail_start_r=TRAIL_START_R, trail_distance_r=SL_DISTANCE_R, max_lookahead=MAX_LOOKAHEAD)

result

,PnL,Cumulative_PnL,Balance
Datetime,,,
2019-12-31 19:00:00-05:00,0.0,0.000000,1000.000000
2019-12-31 19:15:00-05:00,0.0,0.000000,1000.000000
2019-12-31 19:30:00-05:00,0.0,0.000000,1000.000000
2019-12-31 19:45:00-05:00,0.0,0.000000,1000.000000
2019-12-31 20:00:00-05:00,0.0,0.000000,1000.000000
...,...,...,...
2026-08-04 17:45:00-04:00,0.0,504.090964,6040.909637
2026-08-04 18:00:00-04:00,0.0,504.090964,6040.909637
2026-08-04 18:15:00-04:00,0.0,504.090964,6040.909637


### Result plot

In [11]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=result.index,
    y=result["Balance"],
    mode="lines",
    name="Balance"
))

fig.update_layout(
    title="Balance Over Time",
    xaxis_title="Time",
    yaxis_title="Account Balance Over Time ($)",
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

### Stats

In [12]:
def backtest_stats(result_df):
    result_df = result_df.copy()

    # --------------------------------------------------
    # 1. Return %
    # --------------------------------------------------
    initial_balance = result_df["Balance"].iloc[0]

    final_balance = result_df["Balance"].iloc[-1]
    net_profit = final_balance - initial_balance
    return_pct = net_profit / initial_balance * 100

    # --------------------------------------------------
    # 3. Balance Drawdown
    # --------------------------------------------------
    balance_peak = result_df["Balance"].cummax()

    balance_dd = balance_peak - result_df["Balance"]
    balance_dd_pct = balance_dd / balance_peak * 100

    max_balance_dd = balance_dd.max()
    max_balance_dd_pct = balance_dd_pct.max()


    # --------------------------------------------------
    # 3. Win rate
    # --------------------------------------------------
    _winning_trades = result_df["PnL"][result_df["PnL"] > 0].count()
    _losing_trades = result_df["PnL"][result_df["PnL"] < 0].count()

    # --------------------------------------------------
    # 3. Win rate
    # --------------------------------------------------
    s = result_df["PnL"][result_df["PnL"] != 0]

    wins = s > 0
    losses = s < 0

    win_streaks = wins.groupby((wins != wins.shift()).cumsum()).sum()
    loss_streaks = losses.groupby((losses != losses.shift()).cumsum()).sum()

    max_win_streak = int(win_streaks.max())
    max_loss_streak = int(loss_streaks.max())

    # --------------------------------------------------
    # Return results
    # --------------------------------------------------
    return {
        "Initial Balance": initial_balance,
        "Final Balance": final_balance,
        "Net Profit": net_profit,
        "Return %": return_pct,

        "Max Balance DD": max_balance_dd,
        "Max Balance DD %": max_balance_dd_pct,

        "Win rate": 100 * _winning_trades/(_winning_trades + _losing_trades),

        "Winning trades": int(_winning_trades),
        "Losing trades": int(_losing_trades),
        "N° trades": int(_losing_trades + _winning_trades),

        "Max win streak": max_win_streak,
        "Max loss streak": max_loss_streak
    }

def print_stats(stats):
    for name, value in stats.items():
        # print(f"{name:20s}: {value:.4f}")
        print(f"{name:20s}: {value}")


In [13]:
print_stats(backtest_stats(result))

Initial Balance     : 1000.0
Final Balance       : 6040.909637460276
Net Profit          : 5040.909637460276
Return %            : 504.09096374602757
Max Balance DD      : 108.50847313767645
Max Balance DD %    : 8.408214780860137
Win rate            : 56.023506366307544
Winning trades      : 1144
Losing trades       : 898
N° trades           : 2042
Max win streak      : 5
Max loss streak     : 7


### 7. (Optional) Parameter sweep

In [14]:
# from itertools import product

# range_lengths = [i for i in range(1, 6)]
# rrrs = [i * 0.5 for i in range(1, 7)]
# times = ["09:00", "09:15", "09:30", "09:45", "10:00", "10:15", "10:30", "10:45"]
# sl_range_pcts = [0.5, 1, 1.5, 2]

# sl_distance_rs = [i * 0.1 for i in range(3, 11)]
# trail_start_rs = [0.5, 1, 1.5]
# max_lookaheads = [i for i in range(6, 25)]

# results = {}

# for rl, rr, ct, srp, srs, tsr, mlh  in product(range_lengths, rrrs, times, sl_range_pcts, sl_distance_rs, trail_start_rs, max_lookaheads):
#     _result = run_range_breakout(df, range_length=rl, rrr=rr, target_candle_time=ct, sl_range_pct=srp)
#     results[(rl, rr, ct, srp, srs, tsr, mlh)] = backtest_stats(_result)

# best = sorted(
#     results.items(),
#     key=lambda x: x[1]["Max Balance DD %"]
# )[:3]

# # only three best ones
# for (rl, rr, ct, srp, srs, tsr, mlh), backtest_result in best:
#     print(f"Range length: {rl}, RRR: {rr}, Candle time: {ct}, SL range percentage: {srp}, SL Distance_R: {srs}, Trail Start_R: {tsr}, Max Lookahead: {mlh}")
#     print_stats(backtest_result)
#     print("\n\n--------------------------------------------------\n\n")

# print("Done!")